# Bangalore House Price Prediction & Web App
Cleaned, step-by-step pipeline using Kaggle data to train a linear model and serve it via Streamlit.

In [108]:
# Load core libraries
import os, urllib.request, json, pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Load the Dataset

In [109]:
# Check for local path first, fallback to download if not found locally
local_path = 'Bengaluru_House_Data.csv'
url = 'https://raw.githubusercontent.com/codebasics/py/master/DataScience/BangloreHomePrices/model/bengaluru_house_prices.csv'

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    print('Downloading raw data...')
    urllib.request.urlretrieve(url, local_path)
    df = pd.read_csv(local_path)

df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [110]:
df.shape

(13320, 9)

In [111]:
# check null values
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [112]:
# Drop descriptive metadata columns that have too many missing values or weak signal
df = df.drop(['area_type', 'availability', 'society', 'balcony'], axis='columns')
df = df.dropna()
df.head()

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00


In [113]:
# drop rows with null values
df = df.dropna()

In [114]:
df['size'].unique()

array(['2 BHK', '4 Bedroom', '3 BHK', '4 BHK', '6 Bedroom', '3 Bedroom',
       '1 BHK', '1 RK', '1 Bedroom', '8 Bedroom', '2 Bedroom',
       '7 Bedroom', '5 BHK', '7 BHK', '6 BHK', '5 Bedroom', '11 BHK',
       '9 BHK', '9 Bedroom', '27 BHK', '10 Bedroom', '11 Bedroom',
       '10 BHK', '19 BHK', '16 BHK', '43 Bedroom', '14 BHK', '8 BHK',
       '12 Bedroom', '13 BHK', '18 Bedroom'], dtype=object)

In [115]:
df['bhk'] = df['size'].apply(lambda x:int(x.split(' ')[0]))
df = df.drop('size', axis='columns')
df.head()

,location,total_sqft,bath,price,bhk
0,Electronic City Phase II,1056,2.0,39.07,2
1,Chikka Tirupathi,2600,5.0,120.00,4
2,Uttarahalli,1440,2.0,62.00,3
3,Lingadheeranahalli,1521,3.0,95.00,3
4,Kothanur,1200,2.0,51.00,2


In [116]:
df['total_sqft'].unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      shape=(2067,), dtype=object)

In [117]:
# Resolve Total_sqft
def sqft_con(x):
    t = x.split('-')
    if len(t) == 2:
        return ((float(t[0])+float(t[1]))/2)
    try :
        return float(x)
    except :
        return None


df['total_sqft']=df['total_sqft'].apply(sqft_con)
df.isnull().sum()

location       0
total_sqft    46
bath           0
price          0
bhk            0
dtype: int64

In [118]:
# drop rows with null values 
df = df.dropna()
df.isnull().sum()

location      0
total_sqft    0
bath          0
price         0
bhk           0
dtype: int64

In [119]:
df.shape

(13200, 5)

In [120]:
df['price_per_sqft'] = df['price']*100000/df['total_sqft']
df.head()

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,1200.0,2.0,51.00,2,4250.000000


In [121]:
len(df['location'].unique())

1298

In [122]:
df['location'] = df['location'].apply(lambda x: x.strip())
number_of_locations = df.groupby('location')['location'].agg('count').sort_values(ascending = False)
number_of_locations

location
Whitefield               533
Sarjapur  Road           392
Electronic City          304
Kanakpura Road           264
Thanisandra              235
                        ... 
1 Giri Nagar               1
Kanakapura Road,           1
Kanakapura main  Road      1
Kannur                     1
whitefiled                 1
Name: location, Length: 1287, dtype: int64

In [123]:
# Group low-frequency areas into 'other'
locations_less_than_10 = number_of_locations[number_of_locations<=10]
df['location'] = df['location'].apply(lambda x: 'other' if x in locations_less_than_10 else x)
df.head()

,location,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,1200.0,2.0,51.00,2,4250.000000


In [124]:
# A bedroom is rarely under 300-350 sqft so remove outliers
df = df[(df['total_sqft'] / df['bhk']) >= 300]

In [125]:
# Remove unrealistic bathroom counts (e.g. 2 BHK with 5 bathrooms)
df = df[df['bath'] < (df['bhk'] + 2)]

In [126]:
def remove_pps_outliers(data):
    df_out = pd.DataFrame()
    for _, subdf in data.groupby('location'):
        m = np.mean(subdf['price_per_sqft'])
        s = np.std(subdf['price_per_sqft'])
        reduced = subdf[(subdf['price_per_sqft'] > (m - s)) & (subdf['price_per_sqft'] <= (m + s))]
        df_out = pd.concat([df_out, reduced], ignore_index=True)
    return df_out

df = remove_pps_outliers(df)
df = df.drop('price_per_sqft', axis='columns')

In [127]:
df.shape

(10115, 5)

In [128]:
# One-hot encode locations; drop 'other' column to avoid dummy variable trap
dummies = pd.get_dummies(df['location'], dtype=int)
df_final = pd.concat([df.drop('location', axis='columns'), dummies.drop('other', axis='columns')], axis='columns')
df_final.head()

,total_sqft,bath,price,bhk,1st Block Jayanagar,1st Phase JP Nagar,2nd Phase Judicial Layout,2nd Stage Nagarbhavi,5th Block Hbr Layout,5th Phase JP Nagar,...,Vijayanagar,Vishveshwarya Layout,Vishwapriya Layout,Vittasandra,Whitefield,Yelachenahalli,Yelahanka,Yelahanka New Town,Yelenahalli,Yeshwanthpur
0,2850.0,4.0,428.0,4,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1630.0,3.0,194.0,3,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1875.0,2.0,235.0,3,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1200.0,2.0,130.0,3,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1235.0,2.0,148.0,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Model

In [129]:
# Split features & target
X = df_final.drop('price', axis='columns')
y = df_final['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

# Fit Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)
print('Test Set R2 Score:', round(model.score(X_test, y_test), 4))

Test Set R2 Score: 0.8087


/opt/homebrew/Caskroom/miniconda/base/envs/ai_env/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/homebrew/Caskroom/miniconda/base/envs/ai_env/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/homebrew/Caskroom/miniconda/base/envs/ai_env/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [130]:
import pickle, json

# Save the trained model
with open('bangalore_home_prices_model.pickle', 'wb') as f:
    pickle.dump(model, f)

# Save exact column structure needed for frontend inference
columns = {'data_columns': [col.lower() for col in X.columns]}
with open('columns.json', 'w') as f:
    json.dump(columns, f)

print("Export complete: 'bangalore_home_prices_model.pickle' and 'columns.json' created.")

Export complete: 'bangalore_home_prices_model.pickle' and 'columns.json' created.


# Inference

In [131]:
# Quick sanity check helper function
def estimate_price(location, sqft, bath, bhk):
    loc_index = np.where(X.columns == location)[0]
    x_input = np.zeros(len(X.columns))
    x_input[0] = sqft
    x_input[1] = bath
    x_input[2] = bhk
    if len(loc_index) > 0:
        x_input[loc_index[0]] = 1
    return float(model.predict([x_input])[0])

print('Sample prediction (1st Phase JP Nagar, 1000 sqft, 2 bath, 2 BHK):')
print(f'{estimate_price("1st Phase JP Nagar", 1000, 2, 2):.2f} Lakhs')

Sample prediction (1st Phase JP Nagar, 1000 sqft, 2 bath, 2 BHK):
85.72 Lakhs


/opt/homebrew/Caskroom/miniconda/base/envs/ai_env/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# Host the Streamlit App

In [134]:
!streamlit run app.py & npx -y localtunnel --port 8501


      👋 Welcome to Streamlit!

      If you'd like to receive helpful onboarding emails, news, offers, promotions,
      and the occasional swag, please enter your email address below. Otherwise,
      leave this field blank.

      Email: 2026-10-01 07:47:30.231 
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼^C
